# Zomato RAG data preparation

This notebook prepares a fact-grounded restaurant corpus. It statistically imputes only the numeric `rate` feature and exposes that decision through `is_rate_imputed`. Text/list fields are never imputed: missing values remain empty and are represented by availability metadata.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

SOURCE_CSV = Path(r'E:\zomato.csv')
OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'zomato_rag_cleaned.csv'

if not SOURCE_CSV.exists():
    raise FileNotFoundError(f'Update SOURCE_CSV to your dataset location: {SOURCE_CSV}')

raw = pd.read_csv(SOURCE_CSV)
raw.shape, raw.columns.tolist()

In [ ]:
# Keep only fields that contribute grounded restaurant facts.
KEEP_COLUMNS = [
    'name', 'location', 'rest_type', 'cuisines', 'dish_liked', 'reviews_list',
    'approx_cost(for two people)', 'rate', 'votes', 'online_order', 'book_table',
    'menu_item', 'listed_in(type)'
]
missing = sorted(set(KEEP_COLUMNS) - set(raw.columns))
if missing:
    raise KeyError(f'Missing expected columns: {missing}')

df = raw[KEEP_COLUMNS].copy()

# Source values look like '4.1/5', 'NEW', or '-'.  Only genuine numeric ratings
# contribute to the mean used for imputation.
df['rate'] = pd.to_numeric(
    df['rate'].astype('string').str.extract(r'([0-5](?:\.\d+)?)', expand=False),
    errors='coerce'
)
df.loc[~df['rate'].between(0, 5), 'rate'] = np.nan
rating_mean = df['rate'].mean()
df['is_rate_imputed'] = df['rate'].isna()
df['rate'] = df['rate'].fillna(rating_mean).round(2)

# Clean numeric fields without inventing values for missing data.
df['votes'] = pd.to_numeric(df['votes'], errors='coerce').astype('Int64')
df['approx_cost(for two people)'] = pd.to_numeric(
    df['approx_cost(for two people)'].astype('string').str.replace(',', '', regex=False),
    errors='coerce'
).astype('Int64')

rating_mean, int(df['is_rate_imputed'].sum())

In [ ]:
TEXT_COLUMNS = ['name', 'location', 'rest_type', 'cuisines', 'dish_liked', 'reviews_list', 'menu_item', 'listed_in(type)']

def clean_text(value):
    """Normalize whitespace; make null-like list placeholders truly empty."""
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in {'', '[]', 'nan', 'none', 'null'}:
        return ''
    return re.sub(r'\s+', ' ', text)

for column in TEXT_COLUMNS:
    df[column] = df[column].map(clean_text)

# Explicit missingness metadata lets the chatbot state that facts are unavailable.
for column, flag in {
    'cuisines': 'cuisines_available',
    'dish_liked': 'dish_liked_available',
    'reviews_list': 'reviews_available',
    'menu_item': 'menu_available',
}.items():
    df[flag] = df[column].ne('')

# The embedding text is assembled only from present, source-backed facts.
def format_value(value):
    return '' if pd.isna(value) else str(value)

def make_document(row):
    facts = [f"Restaurant: {row['name']}"]
    for label, column in [
        ('Location', 'location'), ('Restaurant type', 'rest_type'), ('Cuisines', 'cuisines'),
        ('Popular dishes', 'dish_liked'), ('Service type', 'listed_in(type)')
    ]:
        if row[column]:
            facts.append(f'{label}: {row[column]}')
    if not pd.isna(row['approx_cost(for two people)']):
        facts.append(f"Approximate cost for two: {int(row['approx_cost(for two people)'])}")
    facts.append(f"Rating: {row['rate']:.2f}/5" + (' (imputed from mean available rating)' if row['is_rate_imputed'] else ''))
    if not pd.isna(row['votes']):
        facts.append(f"Votes: {int(row['votes'])}")
    facts.append(f"Online ordering: {row['online_order']}; Table booking: {row['book_table']}")
    if row['menu_available']:
        facts.append(f"Menu items: {row['menu_item']}")
    if row['reviews_available']:
        facts.append(f"Reviews: {row['reviews_list']}")
    return '\n'.join(facts)

df['rag_document'] = df.apply(make_document, axis=1)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Saved {len(df):,} rows to {OUTPUT_CSV}')
df[['name', 'rate', 'is_rate_imputed', 'menu_available', 'reviews_available', 'rag_document']].head(2)

In [ ]:
# Compact quality checks: no invented text and a transparent rating-imputation audit.
assert set(KEEP_COLUMNS).issubset(df.columns)
assert df['rate'].notna().all()
assert df.loc[~df['menu_available'], 'menu_item'].eq('').all()
assert df.loc[~df['reviews_available'], 'reviews_list'].eq('').all()

print({
    'rows': len(df),
    'mean_rating_used_for_imputation': round(float(rating_mean), 3),
    'imputed_ratings': int(df['is_rate_imputed'].sum()),
    'records_with_menu': int(df['menu_available'].sum()),
    'records_with_reviews': int(df['reviews_available'].sum()),
})